# Additional End of week Exercise - week 2

In [37]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from artists import tools, handle_tool_calls

In [41]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if not(openai_api_key):
    print("OpenAI API Key not set")

LAMA_MODEL = "llama3.2"
DEEPSEEK_MODEL = "deepseek-r1:1.5b"
GPT_MODEL = "gpt-4.1-mini"
OLLAMA_BASE_URL = "http://localhost:11434"

openai = OpenAI()
hasOllama = requests.get(OLLAMA_BASE_URL).content
print(f"Ollama is running: {hasOllama}")

ollama = OpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key='ollama')

Ollama is running: b'Ollama is running'


In [ ]:
system_message = """You are a helpful assistant, working for the Metropolitan Museum of Art in New York. You provide information about artists and shows images of their artworks. 
The tools give you access to a dataset of artists, including their names, the period in which he/she lived, and the collections their work belongs to. 
From the collections, you can infer the style and period of the artists' works.
When asked about an artist you will provide relevant information based on the dataset and use the tools you get to show an image of one piece of the artist's artworks. 
If you don't have information about a specific artist or collection, you will politely inform the user that you don't have that information.
When appropriate, you can by yourself suggest an artist from the dataset that matches the user's interests,or that of a similar style or period. You can also suggest a totally different artist, 
when you think that's appropriate, but explain that it is a different style or period.
"""

In [44]:
system_message = """You are a helpful assistant, working for the Metropolitan Museum of Art in New York. You provide information about artists. 
The tools that are provided to you in this chat give you access to a dataset with information about artists, including their names, the period in which he/she lived, and the collections their work belongs to. 
Keep in mind that the user does not always asks for an artist but also can start a normal social conversation, like a greeting or friendly talk, etcera.
From the collections, you can infer the style and period of the artists' works.
When asked about an artist you will provide relevant information based on the dataset. 
If you don't have information about a specific artist or collection, you will politely inform the user that you don't have that information.
When appropriate, you can suggest an artist from the dataset that matches the user's interests,or that of a similar style or period. You can also suggest a random artist by using the tools, 
when you think that's appropriate, but explain that it is a different style or period.
Only stick to the artists and the information about them you can get by using the tools! Don't use other information you have in your internal dataset. Do not make up any information, if you don't know, say you don't know.
"""

In [39]:
def chat_GPT(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=GPT_MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=GPT_MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [42]:
def chat_ollama(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=DEEPSEEK_MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=DEEPSEEK_MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [45]:
gr.ChatInterface(fn=chat_GPT, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


Random artist: Salvator Rosa
Artist details: Salvator Rosa, period in which he/she lived: (1615–1673), in collection: European paintings.
Found artist: Salvator Rosa
Artist details: Salvator Rosa, period in which he/she lived: (1615–1673), in collection: European paintings.
Random artist: Jean-Baptiste Carpeaux
Artist details: Jean-Baptiste Carpeaux, period in which he/she lived: (1827–1875), in collection: European sculpture and decorative arts.
Found 161 artists within 50 years tolerance for '1827–1875'
Selected contemporary artist: Jean-Baptiste-Siméon Chardin
Artist details: Jean-Baptiste-Siméon Chardin, period in which he/she lived: (1699–1779), in collection: European paintings.
Found 161 artists within 50 years tolerance for '1827–1875'
Selected contemporary artist: Martin Carlin
Artist details: Martin Carlin, period in which he/she lived: (ca. 1730–1785), in collection: European sculpture and decorative arts.
